# Notebook 02 — Limpieza y perfilado

Segundo sub-bloque del Tema 04. Tomas los DataFrames que extrajiste en el Notebook 01 y los **conoces** primero (perfilado) y los **arreglas** después (limpieza). Antes de transformar hacia el modelo dimensional, hay que saber qué problemas tienen los datos crudos.

Al terminar este notebook deberías poder detectar nulos, duplicados, tipos incorrectos y valores anómalos en un DataFrame, y aplicar las correcciones pertinentes con pandas.

**Contenido de este notebook:**

- [Setup — recuperar los DataFrames del Notebook 01](#setup--recuperar-los-dataframes-del-notebook-01)
- [Perfilado — conocer el DataFrame antes de tocarlo](#perfilado--conocer-el-dataframe-antes-de-tocarlo)
- [Manejo de nulos](#manejo-de-nulos)
- [Deduplicación](#deduplicación)
- [Normalización de strings](#normalización-de-strings)
- [Conversión de tipos](#conversión-de-tipos)
- [Estandarización con catálogos](#estandarización-con-catálogos)

## Setup — recuperar los DataFrames del Notebook 01

Decisión pedagógica del Tema 04: cada notebook es **auto-contenido** — puedes abrir cualquiera y correrlo sin depender del estado del anterior. Aquí re-creamos el engine y re-extraemos las tablas necesarias.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# Reemplaza con tus valores del Tema 01
AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

df_customers     = pd.read_sql("SELECT * FROM northwind_oltp.customers",     engine)
df_orders        = pd.read_sql("SELECT * FROM northwind_oltp.orders",        engine)
df_order_details = pd.read_sql("SELECT * FROM northwind_oltp.order_details", engine)
df_products      = pd.read_sql("SELECT * FROM northwind_oltp.products",      engine)
df_employees     = pd.read_sql("SELECT * FROM northwind_oltp.employees",     engine)

print(f"customers       {df_customers.shape}")
print(f"orders          {df_orders.shape}")
print(f"order_details   {df_order_details.shape}")
print(f"products        {df_products.shape}")
print(f"employees       {df_employees.shape}")

## Perfilado — conocer el DataFrame antes de tocarlo

Antes de transformar nada, hay que **mirar los datos**. Pandas trae cuatro herramientas básicas para hacerlo rápido:

| Método | Para qué sirve |
|---|---|
| `df.info()` | Tipos por columna + conteo de no-nulos + uso de memoria |
| `df.describe()` | Estadísticas (count, mean, std, min, max, percentiles) de columnas numéricas |
| `df.isna().sum()` | Cuántos NULL hay en cada columna |
| `df['col'].value_counts()` | Distribución de valores únicos en una columna |

La regla: **antes de cualquier transformación, ejecuta los cuatro** sobre el DataFrame. Es barato (segundos) y te ahorra horas de debuggear con datos que no entendiste bien.

### `df.info()` — tipos y conteo de no-nulos

Te dice:

- Cuántas filas tiene el DataFrame.
- Cuántas columnas y cuáles son.
- El tipo (`Dtype`) que pandas infirió para cada columna.
- Cuántos valores no-nulos hay en cada columna — los que falten respecto al total son NULLs.
- Cuánta memoria está usando.

In [ ]:
df_customers.info()

Fíjate en el `Non-Null Count`: si una columna dice `91 non-null` y la tabla tiene 91 filas, no hay NULLs. Si dice `31 non-null` con 91 filas totales, **60 filas tienen NULL ahí** — eso necesita decisión.

### `df.describe()` — estadísticas numéricas

Por defecto opera sobre **columnas numéricas**. Te muestra count, mean, std, min, percentiles (25/50/75), max. Útil para:

- Detectar **outliers** — `max` muy lejos del percentil 75, o `min` negativo donde no debería.
- Validar rangos plausibles — un precio de venta no puede ser `0.00`, una cantidad no puede ser negativa, etc.

In [ ]:
df_order_details.describe()

Para forzar que incluya **todas** las columnas (incluyendo no-numéricas), usa `include='all'`:

In [ ]:
df_customers.describe(include="all")

Con `include="all"` aparecen cuatro filas adicionales que solo aplican a **columnas no numéricas** — todas hablan de frecuencia:

| Estadística | Qué representa |
|---|---|
| `count`  | Filas no nulas |
| `unique` | Cuántos valores distintos hay |
| `top`    | El valor más frecuente (la moda) |
| `freq`   | Cuántas veces aparece ese valor más frecuente |

Las celdas que quedan como `NaN` son combinaciones que no aplican: `mean`/`std`/percentiles sobre columnas categóricas, o `top`/`freq` sobre columnas numéricas. Por ejemplo, si en `country` lees `top=USA, freq=13`, significa que `USA` es el país que más aparece y lo hace 13 veces — la misma información que daría `df_customers["country"].value_counts().head(1)`, pero presentada junto al resto de estadísticas.

### `df.isna().sum()` — nulos por columna

`df.isna()` devuelve un DataFrame del mismo tamaño con `True` donde hay NULL y `False` donde no. Encadenarlo con `.sum()` cuenta los `True` (porque en aritmética `True == 1`).

In [ ]:
df_customers.isna().sum()

**¿Qué cuenta exactamente como NULL para `isna()`?** Construye una serie con los valores típicos que aparecen en datos sucios y mira qué detecta cada uno — útil para entender qué pasa por debajo:

In [ ]:
import numpy as np

s = pd.Series([
    np.nan,        # NaN (float)
    pd.NA,         # NA (tipos nullable)
    pd.NaT,        # NaT (datetime)
    None,          # None (Python)
    "valor real",
    "",            # string vacío
    "NaN",         # string literal "NaN"
    "null",        # string literal "null"
    "N/A",         # centinela típico
    0,             # cero numérico
])

pd.DataFrame({"valor": s, "isna()": s.isna()})

`isna()` detecta los **cuatro centinelas reales** (`NaN`, `pd.NA`, `NaT`, `None`) pero **ignora los strings que parecen nulos** (`""`, `"NaN"`, `"null"`, `"N/A"`). Esa es la trampa más común al leer datos sucios: un CSV mal exportado pone `"NULL"` como texto literal, `isna()` no los reconoce, y tu reporte de calidad dice "0 nulos" cuando realmente hay miles.

**Antídoto:** al leer la fuente, declara los centinelas que sí debes tratar como nulos:

```python
df = pd.read_csv("archivo.csv", na_values=["NULL", "null", "N/A", "", "NaN"])
```

O, después de leer, conviértelos explícitamente:

```python
df["col"] = df["col"].replace({"N/A": np.nan, "": np.nan})
```

Es el mismo principio que en SQL aplicas con `NULLIF(col, 'N/A')` — convertir el centinela a NULL real antes de procesarlo.

En Northwind verás que `region` y `fax` tienen muchos nulos — los clientes de países sin subdivisión regional, o sin fax. **No es un error de datos, es realidad del negocio** — algunos clientes no tienen esos atributos. La decisión de qué hacer con esos nulos es distinta a si fueran un bug.

Las otras tablas también vale la pena revisarlas:

In [ ]:
df_orders.isna().sum()

Fíjate en `shipped_date` — algunos pedidos tienen NULL ahí, porque **todavía no se habían despachado** al momento del corte de los datos. Lo mismo se reflejará en el DWH: `dim_shipper` y `shipped_date_key` de `fact_sales` aceptan NULL (lo viste en el Tema 02).

### `value_counts()` — distribución de valores únicos

Para columnas categóricas (país, ciudad, categoría, estado), `value_counts()` te muestra cuántas veces aparece cada valor. Es la herramienta clave para detectar:

- **Inconsistencias de capitalización o espacios** (`'Mexico'` vs `'mexico'` vs `' México '` aparecen como tres valores distintos).
- **Cardinalidad** — cuántos valores únicos hay (¿son 5? ¿son 5,000? cambia el enfoque).

In [ ]:
df_customers["country"].value_counts()

In [ ]:
df_orders["ship_country"].value_counts().head(10)

Northwind es un dataset limpio — los nombres de países están estandarizados. En la realidad casi nunca tienes esa suerte; `value_counts()` es donde detectas problemas como `'USA'` / `'U.S.A.'` / `'United States'` apareciendo como tres valores distintos.

## Manejo de nulos

Una vez identificados, hay **tres estrategias** para tratar los nulos. La elección depende del significado de "faltante" en tu dominio.

### Estrategia A — `dropna()`: eliminar filas con nulos

Útil cuando una fila sin el atributo crítico **no sirve** para análisis. Por ejemplo, si tu reporte requiere agrupar por país y un cliente no tiene país registrado, esa fila sobra.

In [ ]:
# Eliminar filas donde country es NULL (no aplica en Northwind, pero ilustrativo)
df_con_country = df_customers.dropna(subset=["country"])
print(f"antes:   {len(df_customers)}")
print(f"después: {len(df_con_country)}")

**Cuidado:** `dropna()` sin parámetros elimina filas con NULL en **cualquier** columna — fácil destruir el 90% del dataset accidentalmente. Siempre usa `subset=[...]` para acotar.

### Estrategia B — `fillna()`: imputar un valor

Útil cuando el campo es importante para análisis pero NULL no aporta. Sustituyes con un valor neutro: `'Unknown'` para texto, `0` para numérico, la media o mediana si la imputación estadística tiene sentido.

In [ ]:
df_customers_clean = df_customers.copy()
df_customers_clean["region"] = df_customers_clean["region"].fillna("N/A")
df_customers_clean["region"].value_counts(dropna=False).head()

### Estrategia C — Indicador binario

A veces "faltante" es información en sí mismo: que un cliente **no tenga fax** dice algo sobre el cliente. En ese caso, **antes** de imputar, agrega una columna binaria que marca dónde había NULL. Después imputa el valor original.

In [ ]:
df_customers_clean["has_fax"] = df_customers_clean["fax"].notna()
df_customers_clean["fax"]     = df_customers_clean["fax"].fillna("N/A")
df_customers_clean[["customer_id", "company_name", "fax", "has_fax"]].head()

Ahora puedes analizar *"¿qué tan distintos son los clientes con vs sin fax?"* sin perder el dato original.

> **Regla operativa:** decidir explícitamente, no dejar que pandas decida por ti. NULLs sin tratar se propagan a las queries del DWH y producen totales raros.

## Deduplicación

Un duplicado es una fila que aparece dos o más veces — habitualmente bug de extracción o de algún join previo mal hecho. En pandas:

- `df.duplicated()` — Serie booleana, `True` si la fila ya apareció antes.
- `df.duplicated().sum()` — cuántos duplicados hay.
- `df.drop_duplicates()` — devuelve el DataFrame sin duplicados.

In [ ]:
duplicados = df_customers.duplicated().sum()
print(f"filas duplicadas en customers: {duplicados}")

Northwind tiene PKs en todas sus tablas, así que no hay duplicados reales. Pero el patrón importa cuando trabajas con fuentes sin PK o con archivos planos.

### `subset` y `keep` — controlar qué cuenta como duplicado

- **`subset=[...]`** — define duplicado solo en base a esas columnas, ignorando el resto.
- **`keep='first'`** (default) — conserva la primera ocurrencia.
- **`keep='last'`** — conserva la última.
- **`keep=False`** — elimina **todas** las ocurrencias (útil cuando quieres descartar cualquier valor ambiguo).

Ejemplo: simular duplicados y deduplicar por `customer_id`:

In [ ]:
# Construir un caso con duplicados
df_con_dupes = pd.concat([df_customers.head(3), df_customers.head(3)], ignore_index=True)
print(f"shape con duplicados:    {df_con_dupes.shape}")

df_dedupe = df_con_dupes.drop_duplicates(subset=["customer_id"], keep="first")
print(f"shape sin duplicados:    {df_dedupe.shape}")

> **Nota histórica — `DataFrame.append` ya no existe.**
>
> Hasta pandas 1.x existía `df.append(otro_df)` como método para concatenar DataFrames. Era equivalente conceptual a `pd.concat`, pero menos eficiente: creaba una copia completa en cada llamada y no podía concatenar varios en una sola pasada.
>
> - **Deprecación:** pandas **1.4.0** (22 de enero de 2022) marcó `DataFrame.append` y `Series.append` como deprecated con un `FutureWarning`.
> - **Eliminación:** pandas **2.0.0** (3 de abril de 2023) los removió por completo.
>
> Si lo encuentras en código antiguo, el reemplazo directo es `pd.concat([df1, df2], ignore_index=True)` — soporta listas arbitrarias de DataFrames en una sola llamada y es el patrón oficial desde entonces.

## Normalización de strings

El accesor `.str` de pandas aplica operaciones de string **vectorizadas** sobre una Serie completa — más rápido que iterar fila por fila y la sintaxis es declarativa.

Las cuatro operaciones más comunes:

| Operación | Para qué |
|---|---|
| `str.strip()` | Quitar espacios al inicio y al fin |
| `str.lower()` / `str.upper()` / `str.title()` | Uniformar capitalización |
| `str.replace(...)` | Sustituir un patrón (string o regex) |
| `str.contains(...)` | Filtrar filas cuyo string matchea un patrón |

In [ ]:
# Antes: ' Mexico ', 'mexico', 'MEXICO' aparecen como 3 valores distintos
ejemplo = pd.Series([" Mexico ", "mexico", "MEXICO", "Mexico"])
print("antes:")
print(ejemplo.value_counts())

ejemplo_normalizado = ejemplo.str.strip().str.title()
print("\ndespués:")
print(ejemplo_normalizado.value_counts())

Con regex también puedes hacer reemplazos más sofisticados:

In [ ]:
# Eliminar caracteres no alfabéticos de los teléfonos
df_customers["phone_limpio"] = (
    df_customers["phone"]
    .str.replace(r"[^\d]", "", regex=True)
)
df_customers[["phone", "phone_limpio"]].head()

> **Recurso útil — [regexr.com](https://regexr.com/)**
>
> Escribir regex "en frío" es engorroso: un caracter mal puesto cambia el patrón completo. [regexr.com](https://regexr.com/) te deja escribir el patrón a la izquierda, pegar texto de prueba a la derecha, y ver en vivo qué partes matchean — con explicación token por token (qué hace `\d`, qué hace `[^...]`, qué hace `+` vs `*`, etc.).
>
> Para los ejemplos del módulo (limpieza de teléfonos, parsing de `bathrooms_text` de Airbnb, normalización de strings), pegar primero la muestra real y construir el patrón ahí es mucho más rápido que iterar a ciegas en Python.

## Conversión de tipos

pandas **infiere** tipos al leer un DataFrame, pero la inferencia se equivoca con frecuencia — especialmente con dinero (lo lee como `float64`, no como `Decimal`) y con fechas (las deja como `object` si vienen sin formato evidente).

### `astype` — conversión simple

Para forzar un tipo numérico o categórico:

In [ ]:
print("antes:")
print(df_orders.dtypes)

df_orders["customer_id"] = df_orders["customer_id"].astype("string")
df_orders["employee_id"] = df_orders["employee_id"].astype("Int16")  # Int16 nullable, no int16

print("\ndespués:")
print(df_orders[["customer_id", "employee_id"]].dtypes)

Nota: `Int16` (con mayúscula) es el tipo **nullable** de pandas, que sí acepta NaN; `int16` (minúscula) no acepta nulos y falla si hay alguno. En ETL casi siempre quieres el nullable.

### `pd.to_datetime` — parseo de fechas

Para columnas de fecha, usa `pd.to_datetime` con `errors='coerce'` — los valores que no se puedan parsear se vuelven `NaT` (Not a Time) en vez de tirar excepción.

In [ ]:
df_orders["order_date"]    = pd.to_datetime(df_orders["order_date"],    errors="coerce")
df_orders["required_date"] = pd.to_datetime(df_orders["required_date"], errors="coerce")
df_orders["shipped_date"]  = pd.to_datetime(df_orders["shipped_date"],  errors="coerce")

df_orders[["order_date", "required_date", "shipped_date"]].dtypes

Con las fechas como `datetime64[ns]`, ahora puedes operar sobre ellas: extraer año, mes, día de semana, calcular diferencias, etc.

### Dinero: cuidado con `float`

El precio en `order_details` viene como `REAL` (float aproximado) en el OLTP. Lo viste en el Tema 02: para sumas analíticas masivas, el error de redondeo binario se acumula. En el DWH lo convertimos a `NUMERIC(10,2)`.

En pandas el equivalente es usar `Decimal` o redondear explícitamente al convertir. Para este módulo, **redondeo explícito** al cargar es suficiente:

In [ ]:
df_order_details["unit_price"] = df_order_details["unit_price"].round(2)
df_order_details["discount"]   = df_order_details["discount"].round(2)
df_order_details.head(3)

## Estandarización con catálogos

Cuando un campo tiene un **dominio cerrado** de valores válidos (categorías, países, estados de pedido), aplicas un **catálogo de valores controlados**: un diccionario que mapea las variantes que aparezcan en los datos al valor canónico.

Patrón:

In [ ]:
# Ejemplo: estandarizar variantes de país a la forma canónica
mapeo_paises = {
    "USA":            "United States",
    "U.S.A.":         "United States",
    "US":             "United States",
    "México":         "Mexico",
    "MX":             "Mexico",
    "UK":             "United Kingdom",
    "Gran Bretaña":   "United Kingdom",
}

# Simular un campo con variantes
ejemplo = pd.Series(["USA", "U.S.A.", "Mexico", "MX", "UK", "Argentina"])
ejemplo_canonico = ejemplo.replace(mapeo_paises)
print(ejemplo_canonico.tolist())

### Detectar valores fuera del catálogo

Importante: si el catálogo es **cerrado** (solo aceptas los valores que conoces), debes detectar y reportar los que aparezcan fuera. No los descartes silenciosamente — podrías estar perdiendo datos legítimos.

In [ ]:
catalogo_valido = {"United States", "Mexico", "United Kingdom"}
fuera = ejemplo_canonico[~ejemplo_canonico.isin(catalogo_valido)]

if len(fuera) > 0:
    print("Valores fuera del catálogo:")
    print(fuera.tolist())
else:
    print("Todos los valores están en el catálogo.")

## Cierre

Los DataFrames ya están **perfilados, limpios y tipados**:

- Sabes cuántos nulos hay, dónde, y qué hacer con ellos (drop / fillna / indicador).
- Sabes detectar y eliminar duplicados.
- Strings normalizados (sin espacios sobrantes, capitalización uniforme).
- Tipos correctos: fechas como `datetime`, dinero redondeado, IDs como nullable integer.
- Campos categóricos estandarizados contra un catálogo de valores controlados.

Listos para la fase de **transformación según reglas de negocio**. El siguiente notebook (**03 — Transformación**) toma estos DataFrames y construye las dimensiones desnormalizadas + la tabla de hechos.

---

<p align="center">
<a href="01_fundamentos_y_extraccion.ipynb">← Anterior: Notebook 01</a> | <a href="Readme.md">Volver al índice</a> | <a href="03_transformacion.ipynb">Siguiente: Notebook 03 — Transformación →</a>
</p>